# On-Policy Distillation — Qwen3 Reasoning Variant

**Context-distillation with thinking models.**
A small thinking **student** (`Qwen3-0.6B`) knows nothing about the AC manual.
A larger frozen thinking **teacher** (`Qwen3-4B-Thinking-2507`) also doesn't — but if we
paste the manual into the teacher's prompt context, it answers well.
On-policy distillation moves that ability into the student's weights
*without the student ever seeing the manual*.

Both models generate `<think>...</think>` blocks before answering, so the distillation loss
runs over dense reasoning tokens — the filler dilution problem from the LFM2 run is largely gone.

```
              question
                 │
    ┌────────────┴──────────────┐
    ▼                           ▼
STUDENT prompt             TEACHER prompt
(no manual)                (manual in context)
both end with <think>\n    both end with <think>\n
    │ sample                     │
    ▼                            │
<think>                          │
  wrong reasoning...             │
</think>                         │
wrong answer        ────────────►│  score SAME tokens (frozen)
                                 ▼
              reverse KL: dense signal over every reasoning step
                                 │
                    backprop into student LoRA
```

**Runtime:** Colab → Runtime → Change runtime type → **L4 GPU**  
**Manual:** Upload `sharp_cv_p09fx_manual_en.md` to Colab before running.
See the README for how to convert the PDF to a clean English-only Markdown file.


In [ ]:
# 0. Install. After this runs: Runtime → Restart session, then Run all.
# transformers>=4.51 is required for Qwen3 and the enable_thinking parameter.
!pip install -q "transformers>=4.51" "peft>=0.13" accelerate


In [ ]:
import zipfile
with zipfile.ZipFile("qwen3_06b_ac_sft_lora.zip", "r") as z:
    z.extractall("qwen3_06b_ac_sft_lora")

In [ ]:
# 1. Imports + config
import re, random, json
import requests
import torch
import torch.nn.functional as F
from transformers import AutoTokenizer, AutoModelForCausalLM
from peft import LoraConfig, get_peft_model

STUDENT_ID = "Qwen/Qwen3-0.6B"
TEACHER_ID = "Qwen/Qwen3-1.7B"      # was 4B — 1.7B is also a thinking model, ~2x faster

TRAIN_JSONL_URL = "ac_manual_synth_tra.jsonl"
VAL_JSONL_URL   = "ac_manual_synth_val.jsonl"
MANUAL_MD_PATH  = "sharp_cv_p09fx_manual_en.md"

SYSTEM = ("You are a helpful assistant for the Sharp CV-P09FX portable air conditioner. "
          "Answer questions accurately based on the product manual.")

# Training knobs
EPOCHS         = 2      # SFT warmstart means less distillation needed; plateau by step ~30
MAX_NEW_TOKENS = 256    # _trim_at_think_end stops earlier anyway; 256 caps worst case
N_DISTILL_QS   = 60     # fact-dense subset of train questions used for distillation
GEN_TEMP       = 1.0
GEN_TOP_P      = 0.95
KL_TEMP        = 1.0
LR             = 1e-4
ACCUM_STEPS    = 4
SEED           = 0

DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
DTYPE  = torch.bfloat16 if DEVICE == "cuda" else torch.float32
random.seed(SEED); torch.manual_seed(SEED)
print("device:", DEVICE, "| dtype:", DTYPE)


In [ ]:
# 2. Data loaders
_USER_RE = re.compile(r"<\|im_start\|>user\n(.*?)<\|im_end\|>", re.DOTALL)

def load_questions(url):
    """Each JSONL line is {"text": "<ChatML conversation>"}; extract the user question."""
    text = requests.get(url, timeout=60).text
    qs = []
    for line in text.splitlines():
        line = line.strip()
        if not line:
            continue
        m = _USER_RE.search(json.loads(line)["text"])
        if m:
            qs.append(m.group(1).strip())
    return qs

def load_manual():
    """Load the cleaned English-only Markdown manual from disk.
    This replaces the PDF extraction + keyword-filter workaround from the LFM2 run.
    The MD file is version-controlled and you can verify its contents directly."""
    with open(MANUAL_MD_PATH, encoding="utf-8") as f:
        return f.read()


In [ ]:
# 3. Tokenizer + models
tokenizer = AutoTokenizer.from_pretrained(STUDENT_ID)
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token

# Safety check: token-level KL is only valid if both models share the same vocab mapping.
t_tok = AutoTokenizer.from_pretrained(TEACHER_ID)
probe = "minimum window width 22 inches 559mm installation"
assert tokenizer(probe)["input_ids"] == t_tok(probe)["input_ids"], \
    "Student and teacher tokenizers disagree -- token-level KL would be meaningless."
del t_tok
print("Tokenizer alignment OK.")

# Teacher: frozen, no gradients.
# No quantization needed on L4 (24 GB): Qwen3-4B in bf16 ~ 8 GB, well within budget.
teacher = AutoModelForCausalLM.from_pretrained(TEACHER_ID, torch_dtype=DTYPE).to(DEVICE).eval()
for p in teacher.parameters():
    p.requires_grad_(False)

# Student: LoRA on attention projections only.
student = AutoModelForCausalLM.from_pretrained(STUDENT_ID, torch_dtype=DTYPE).to(DEVICE)
student = get_peft_model(student, LoraConfig(
    r=16, lora_alpha=32, lora_dropout=0.0,
    target_modules=["q_proj", "k_proj", "v_proj", "o_proj"],  # Qwen3 attention projections
    task_type="CAUSAL_LM",
))
student.print_trainable_parameters()


In [ ]:
# 4. Prompt builders + rollout

def build_prompt_ids(question, with_manual, manual_text=None):
    sys_content = SYSTEM
    if with_manual and manual_text:
        sys_content = SYSTEM + "\n\nProduct manual:\n" + manual_text
    messages = [{"role": "system", "content": sys_content},
                {"role": "user",   "content": question}]
    ids = tokenizer.apply_chat_template(
        messages,
        add_generation_prompt=True,
        enable_thinking=True,
        return_tensors="pt",
        return_dict=True,
    )["input_ids"]
    return ids.to(DEVICE)

def _trim_at_eos(gen):
    """Trim gen tokens after first EOS. Handles Qwen3's eos_token_id being a list."""
    eos_ids = tokenizer.eos_token_id
    if eos_ids is None:
        return gen
    if not isinstance(eos_ids, list):
        eos_ids = [eos_ids]
    for pos in range(gen.shape[1]):
        if gen[0, pos].item() in eos_ids:
            return gen[:, :pos + 1]
    return gen

# Token ID for </think>. Qwen3 registers it as a special token (ID 151669).
# We resolve it at runtime so this works even if the ID ever changes.
THINK_END_ID = tokenizer.convert_tokens_to_ids("</think>")
if THINK_END_ID == tokenizer.unk_token_id:
    _ids = tokenizer.encode("</think>", add_special_tokens=False)
    THINK_END_ID = _ids[0] if _ids else None
print(f"</think> token id: {THINK_END_ID}")

def _trim_at_think_end(gen):
    """Trim rollout at the first </think> token (inclusive).
    Prevents training on looping sequences that never terminate the think block.
    KL is then computed only over the reasoning tokens, which is where the
    fact-learning signal lives anyway."""
    if THINK_END_ID is None:
        return gen
    for pos in range(gen.shape[1]):
        if gen[0, pos].item() == THINK_END_ID:
            return gen[:, :pos + 1]
    return gen  # </think> not found — rollout hit MAX_NEW_TOKENS without terminating

@torch.no_grad()
def student_rollout(question):
    """Sample one on-policy rollout from the student (no manual).
    Rollout is trimmed at the first </think> token so the distillation loss
    is computed only over thinking tokens — the model is never trained to loop."""
    prompt = build_prompt_ids(question, with_manual=False)
    out = student.generate(
        prompt, attention_mask=torch.ones_like(prompt),
        max_new_tokens=MAX_NEW_TOKENS, do_sample=True,
        temperature=GEN_TEMP, top_p=GEN_TOP_P,
        pad_token_id=tokenizer.pad_token_id,
    )
    gen = _trim_at_think_end(out[:, prompt.shape[1]:])
    return prompt, gen

def logits_on_gen(model, prompt_ids, gen_ids):
    """Run model on [prompt ++ gen]; return logits predicting each gen token: [gen_len, V]."""
    full   = torch.cat([prompt_ids, gen_ids], dim=1)
    logits = model(full).logits[0]
    start  = prompt_ids.shape[1] - 1
    return logits[start : start + gen_ids.shape[1]]


In [ ]:
# 5. Distillation helpers

FORMAT_PENALTY = 0.5   # added to KL when rollout never reaches </think>

def collect_rollouts(questions, manual_text):
    """Generate student rollouts for all questions and pre-compute teacher logits.
    Teacher runs ONCE here per epoch — zero teacher forward passes during gradient steps.
    Returns list of (prompt_s, gen, t_logits) tuples, skipping empty rollouts."""
    was_training = student.training
    student.eval()
    cache = []
    n = len(questions)
    for idx, q in enumerate(questions):
        prompt_s, gen = student_rollout(q)
        if gen.shape[1] == 0:
            continue
        with torch.no_grad():
            prompt_t = build_prompt_ids(q, with_manual=True, manual_text=manual_text)
            t_logits = logits_on_gen(teacher, prompt_t, gen).float()
        cache.append((prompt_s, gen, t_logits))
        if (idx + 1) % 10 == 0 or idx == n - 1:
            print(f"  collected {idx + 1}/{n}", end="\r")
    print()
    if was_training:
        student.train()
    return cache

def distill_step(prompt_s, gen, t_logits):
    """KL loss using pre-cached teacher logits. No teacher forward pass needed."""
    s_logits = logits_on_gen(student, prompt_s, gen).float()
    s_logp = F.log_softmax(s_logits / KL_TEMP, dim=-1)
    t_logp = F.log_softmax(t_logits / KL_TEMP, dim=-1)
    kl = (s_logp.exp() * (s_logp - t_logp)).sum(dim=-1).mean()
    terminated = (THINK_END_ID is not None and gen[0, -1].item() == THINK_END_ID)
    penalty = 0.0 if terminated else FORMAT_PENALTY
    return kl + penalty


In [ ]:
# 6. Eval helpers
@torch.no_grad()
def answer(question):
    """Greedy student answer (no manual). skip_special_tokens=True removes <think>/<|im_end|>
    tokens but keeps the reasoning text, so you can see whether the student is reasoning."""
    prompt = build_prompt_ids(question, with_manual=False)
    out = student.generate(
        prompt, attention_mask=torch.ones_like(prompt),
        max_new_tokens=768, do_sample=False,
        pad_token_id=tokenizer.pad_token_id,
        repetition_penalty=1.15,
    )
    return tokenizer.decode(out[0, prompt.shape[1]:], skip_special_tokens=True).strip()

@torch.no_grad()
def eval_kl(questions, manual_text):
    """Mean per-token reverse KL on held-out questions using GREEDY rollouts.
    Deterministic, so Before vs After is a fair comparison. Lower = closer to teacher."""
    was_training = student.training
    student.eval()
    total_kl = 0.0; total_tok = 0
    for q in questions:
        prompt_s = build_prompt_ids(q, with_manual=False)
        out = student.generate(prompt_s, attention_mask=torch.ones_like(prompt_s),
                               max_new_tokens=MAX_NEW_TOKENS, do_sample=False,
                               pad_token_id=tokenizer.pad_token_id)
        gen = _trim_at_eos(out[:, prompt_s.shape[1]:])
        if gen.shape[1] == 0:
            continue
        prompt_t = build_prompt_ids(q, with_manual=True, manual_text=manual_text)
        s_logp = F.log_softmax(logits_on_gen(student, prompt_s, gen).float() / KL_TEMP, dim=-1)
        t_logp = F.log_softmax(logits_on_gen(teacher, prompt_t, gen).float() / KL_TEMP, dim=-1)
        kl = (s_logp.exp() * (s_logp - t_logp)).sum(dim=-1)
        total_kl += kl.sum().item(); total_tok += kl.shape[0]
    if was_training:
        student.train()
    return total_kl / max(total_tok, 1)


In [ ]:
# 7. Load data + manual
questions  = load_questions(TRAIN_JSONL_URL)
val_qs     = load_questions(VAL_JSONL_URL)
manual     = load_manual()
kl_eval_qs = val_qs[:20]

print(f"Train questions : {len(questions)}")
print(f"Val questions   : {len(val_qs)}")
print(f"Manual tokens   : {len(tokenizer(manual)['input_ids'])}")
print(f"'559' in manual : {'559' in manual}")

# Fact-dense subset for distillation: questions that mention measurements, parts, or
# procedures. These give the richest KL signal. Fill up to N_DISTILL_QS with remaining
# questions if the keyword filter returns fewer than needed.
_FACT_KW = ["22", "559", "inch", "mm", "screw", "drain", "install", "width",
            "window", "exhaust", "filter", "reset", "power", "hose", "panel"]

def _is_fact_dense(q):
    q_l = q.lower()
    return any(kw in q_l for kw in _FACT_KW)

distill_qs  = [q for q in questions if _is_fact_dense(q)]
_remaining  = [q for q in questions if not _is_fact_dense(q)]
distill_qs  = (distill_qs + _remaining)[:N_DISTILL_QS]

print(f"\nDistillation questions : {len(distill_qs)} "
      f"(fact-dense first, padded from remaining if needed)")


## Teacher probe

**Run this before training.** Lesson from the LFM2 run: the student can only become as
good as the teacher — if the teacher answers wrong, you distill wrong reasoning.

Expect the teacher to produce a `<think>` block that reasons from the manual and arrives
at **22 inches (559mm)**. If it doesn't, fix the manual file before proceeding.


In [ ]:
# Verify teacher answers correctly before training.
probe_q = "What is the minimum window width required for the window panel installation?"
pt = build_prompt_ids(probe_q, with_manual=True, manual_text=manual)
out = teacher.generate(pt, max_new_tokens=256, do_sample=False,
                       pad_token_id=tokenizer.pad_token_id)
print(tokenizer.decode(out[0, pt.shape[1]:], skip_special_tokens=True))
# Expect: reasoning that references the manual, concluding with ~"22 inches (559mm)"


## Before training

Baseline held-out reverse-KL and sample answers. The answers will be wrong — that's expected.
The think blocks show the student's reasoning before any distillation pressure.


In [ ]:
print(f"held-out mean reverse-KL (before): {eval_kl(kl_eval_qs, manual):.4f}")
for q in val_qs[:3]:
    print(f"\nQ: {q}\nA: {answer(q)}")


## Train

Watch `reverseKL` drop each step. Because the student's rollouts include full `<think>` blocks,
the loss covers dense reasoning tokens — expect smoother convergence than the LFM2 run where
signal was diluted by filler tokens.

Re-run this cell to train more epochs.


In [ ]:
opt = torch.optim.AdamW([p for p in student.parameters() if p.requires_grad], lr=LR)
step = 0

for epoch in range(EPOCHS):
    # --- Phase 1: collect rollouts + cache teacher logits (teacher runs once per epoch) ---
    print(f"Epoch {epoch} — collecting {len(distill_qs)} rollouts + caching teacher logits...")
    cache = collect_rollouts(distill_qs, manual)
    random.shuffle(cache)
    print(f"  cached {len(cache)} rollouts")

    # --- Phase 2: gradient steps using cached teacher logits (no teacher forward passes) ---
    student.train()
    running = 0.0; counted = 0
    opt.zero_grad()

    for i, (prompt_s, gen, t_logits) in enumerate(cache):
        loss = distill_step(prompt_s, gen, t_logits)
        (loss / ACCUM_STEPS).backward()
        running += loss.item(); counted += 1

        if (i + 1) % ACCUM_STEPS == 0:
            torch.nn.utils.clip_grad_norm_(
                [p for p in student.parameters() if p.requires_grad], 1.0)
            opt.step(); opt.zero_grad(); step += 1
            if step % 5 == 0:
                print(f"  step {step}  KL={running/max(counted,1):.4f}")
                running = 0.0; counted = 0

    opt.step(); opt.zero_grad()  # flush remainder
    print(f"Epoch {epoch} done.\n")

print("Training complete.")


## After training

Reverse-KL should be lower. Check whether the student's think blocks now reason toward the
correct specs (22 inches / 559mm) even without the manual in context — that's the signal
that distillation worked on reasoning, not just surface style.


In [ ]:
student.eval()
print(f"held-out mean reverse-KL (after): {eval_kl(kl_eval_qs, manual):.4f}")
for q in val_qs[:3]:
    print(f"\nQ: {q}\nA: {answer(q)}")

out_dir = "./qwen3_06b_ac_onpolicy_lora"
student.save_pretrained(out_dir)
print(f"\nSaved student LoRA adapter to {out_dir}")
